[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TVinhQuang/Frequent_Itemset_Mining/blob/main/colab/Apriori_Demo.ipynb)

# Apriori bằng Julia: demo và phân tích giỏ hàng (Market Basket Analysis)

Notebook này chạy **toàn bộ đồ án** trên Google Colab:

1. Cài Julia, tải mã nguồn, chạy bộ unit test.
2. Chạy Apriori trên ví dụ của bài báo Agrawal & Srikant, so sánh hai bộ đếm `scan` và `hash-tree`.
3. Khai phá frequent itemsets trên **Retail** (88.162 giỏ hàng, cửa hàng bán lẻ ở Bỉ).
4. Sinh **luật kết hợp** `X ⇒ Y` (support, confidence, lift) và trình bày top-10 luật theo lift.
5. Kiểm tra tái lập kết quả.

> **Kernel:** Colab chỉ hỗ trợ kernel Python, nên notebook chạy trên Python và gọi Julia qua lệnh shell (`!julia ...`).
> Chạy các ô theo thứ tự từ trên xuống. Ô cài Julia mất khoảng 1 phút.

## 0. Cài đặt môi trường

In [14]:
!curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o julia.tar.gz
!tar -xzf julia.tar.gz -C /usr --strip-components 1
!julia --version

julia version 1.12.6


In [15]:
# Tải mã nguồn (bỏ qua nếu đã có) rồi chuyển vào thư mục dự án
!test -d Frequent_Itemset_Mining || git clone --depth 1 https://github.com/TVinhQuang/Frequent_Itemset_Mining.git
%cd Frequent_Itemset_Mining
!git log -1 --format='commit %h | %ad' --date=short
!mkdir -p output

Cloning into 'Frequent_Itemset_Mining'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 112 (delta 0), reused 87 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 54.63 MiB | 22.40 MiB/s, done.
/content/Frequent_Itemset_Mining/Frequent_Itemset_Mining
commit 5333f0e | 2026-11-22


In [16]:
# Cài dependency (nếu có) và biên dịch trước package
!julia --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()'

    Updating registry at `~/.julia/registries/General.toml`
    Updating `/content/Frequent_Itemset_Mining/Frequent_Itemset_Mining/Project.toml`
  [ade2ca70] ~ Dates ⇒ v1.11.0
  [de0858da] ~ Printf ⇒ v1.11.0
  [ea8e919c] ~ SHA ⇒ v0.7.0
  [fa267f1f] ~ TOML ⇒ v1.0.3
  [cf7118a7] ~ UUIDs ⇒ v1.11.0
    Updating `/content/Frequent_Itemset_Mining/Frequent_Itemset_Mining/Manifest.toml`
  [ade2ca70] ~ Dates ⇒ v1.11.0
  [de0858da] ~ Printf ⇒ v1.11.0
  [9a3f8284] + Random v1.11.0
  [ea8e919c] ~ SHA ⇒ v0.7.0
  [fa267f1f] ~ TOML ⇒ v1.0.3
  [cf7118a7] ~ UUIDs ⇒ v1.11.0
  [4ec0a83e] + Unicode v1.11.0
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/1
  ◒ FrequentItemsetMining
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/1
  ◐ FrequentItemsetMining
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/1
  ◓ FrequentItemsetMining
Precompiling packages  ╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/1
  ◑ FrequentItemsetMining
Precompiling packages  ╺━━━━━━

In [17]:
# Seed cố định cho mọi phần ngẫu nhiên trong notebook.
# Apriori và bộ sinh luật không dùng số ngẫu nhiên; unit test dùng MersenneTwister(42);
# lấy mẫu của benchmark dùng seed 42 (xem benchmarks/AGENTS.md).
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('seed =', SEED)

seed = 42


## 1. Unit test tự động

Lệnh chuẩn của dự án là `julia --project test/runtests.jl`. Ô thứ hai chạy riêng test của bộ sinh luật kết hợp
(`test/rules_tests.jl`); nếu đã thêm `include("rules_tests.jl")` vào `runtests.jl` thì nó đã được chạy ở ô đầu.

In [18]:
!julia --project=. test/runtests.jl

Test Summary:         | Pass  Total   Time
FrequentItemsetMining | 2842   2842  42.5s
Test Summary:    | Pass  Total  Time
AssociationRules |  236    236  2.1s


In [19]:
!julia --project=. test/rules_tests.jl

Test Summary:    | Pass  Total  Time
AssociationRules |  236    236  2.6s


## 2. Ví dụ của bài báo (4 transaction, min count = 2)

In [20]:
!julia --project=. bin/apriori.jl --input data/toy/paper.txt --output output/paper.txt --minsup 0.5 --counter hash-tree --trace
print('--- itemsets ---')
!cat output/paper.txt

transactions=4 min_count=2 itemsets=9
k=1 candidates=5 frequent=4
k=2 candidates=6 frequent=4
k=3 candidates=1 frequent=1
Report: /content/Frequent_Itemset_Mining/Frequent_Itemset_Mining/output/runs/paper-1
MiningMs: 2558.694385
--- itemsets ---
1 #SUP: 2
2 #SUP: 3
3 #SUP: 3
5 #SUP: 3
1 3 #SUP: 2
2 3 #SUP: 2
2 5 #SUP: 3
3 5 #SUP: 2
2 3 5 #SUP: 2


In [21]:
# Hai bộ đếm phải cho cùng tập kết quả
!julia --project=. bin/apriori.jl --input data/toy/paper.txt --output output/paper-scan.txt --minsup 0.5 --counter scan --quiet
ka = set(open('output/paper.txt').read().splitlines()); kb = set(open('output/paper-scan.txt').read().splitlines())
print('scan == hash-tree:', 'KHỚP' if ka == kb else 'KHÁC', f'({len(ka)} itemsets)')

scan == hash-tree: KHỚP (9 itemsets)


In [22]:
# Luật kết hợp trên ví dụ nhỏ
!julia --project=. bin/rules.jl --itemsets output/paper.txt --data data/toy/paper.txt --minconf 0.6 --top 10

Transaction N = 4 | itemsets = 9 | luật (minconf=0.6, minlift=0.0) = 14
  1. {2} => {5} | count=3 sup=0.75 conf=1.0 lift=1.333
  2. {5} => {2} | count=3 sup=0.75 conf=1.0 lift=1.333
  3. {1} => {3} | count=2 sup=0.5 conf=1.0 lift=1.333
  4. {2, 3} => {5} | count=2 sup=0.5 conf=1.0 lift=1.333
  5. {3, 5} => {2} | count=2 sup=0.5 conf=1.0 lift=1.333
  6. {2} => {3, 5} | count=2 sup=0.5 conf=0.6667 lift=1.333
  7. {3} => {1} | count=2 sup=0.5 conf=0.6667 lift=1.333
  8. {5} => {2, 3} | count=2 sup=0.5 conf=0.6667 lift=1.333
  9. {2} => {3} | count=2 sup=0.5 conf=0.6667 lift=0.889
 10. {2, 5} => {3} | count=2 sup=0.5 conf=0.6667 lift=0.889


## 3. Dữ liệu Retail

Retail là dữ liệu giỏ hàng ẩn danh của một cửa hàng bán lẻ ở Bỉ (88.162 giỏ, 16.470 mặt hàng) trong kho dataset SPMF.
Mỗi dòng là một giỏ hàng, mỗi số là mã một mặt hàng. Tên mặt hàng **không** được công bố.

In [23]:
DATA = 'data/dataset_train/retail.txt'
!mkdir -p data/dataset_train
!test -s {DATA} || curl -fsSL https://www.philippe-fournier-viger.com/spmf/datasets/retail.txt -o {DATA}
!wc -l {DATA}
!sha256sum {DATA}
!head -3 {DATA}

11 data/dataset_train/retail.txt
456845123fe4f3db37b365aea849b25b33705249c6a2ae33e538809f2a563363  data/dataset_train/retail.txt
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html><head>
<title>300 Multiple Choices</title>


## 4. Khai phá frequent itemsets

Các tham số có thể chỉnh ở ô dưới. `MINSUP` càng thấp thì càng nhiều itemset và chạy càng lâu
(Colab chậm hơn máy cá nhân). Nếu muốn nhiều luật lift cao hơn, thử hạ `MINSUP` xuống 0.01.

In [24]:
MINSUP = 0.02      # support tối thiểu (tỷ lệ)
MINCONF = 0.5      # confidence tối thiểu của luật
ITEMSETS = 'output/retail.txt'
RULES_CSV = 'output/retail-rules.csv'

In [25]:
%%time
!julia --project=. bin/apriori.jl --input {DATA} --output {ITEMSETS} --minsup {MINSUP} --counter hash-tree --trace

apriori: error: ArgumentError: line 1: expected nonnegative integer items
CPU times: user 19.9 ms, sys: 5.3 ms, total: 25.2 ms
Wall time: 5.73 s


In [26]:
# Số itemset theo kích thước
from collections import Counter
sizes = Counter(len(line.split('#SUP:')[0].split())
                for line in open(ITEMSETS) if '#SUP:' in line)
for k in sorted(sizes): print(f'{k}-itemset: {sizes[k]:,}')
print('Tổng:', f'{sum(sizes.values()):,}')

FileNotFoundError: [Errno 2] No such file or directory: 'output/retail.txt'

## 5. Sinh luật kết hợp

Với mỗi itemset phổ biến `Z` và mỗi cách tách `Z = X ∪ Y`:

- support = sup(Z) / N
- confidence = sup(Z) / sup(X)
- lift = confidence / (sup(Y) / N)

Chỉ giữ luật có confidence ≥ `MINCONF`. Sup(X) và sup(Y) lấy từ chính file itemsets (tính chất Apriori).

In [ ]:
%%time
!julia --project=. bin/rules.jl --itemsets {ITEMSETS} --data {DATA} --minconf {MINCONF} --top 10 --out {RULES_CSV}

## 6. Top-10 luật theo lift

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rules = pd.read_csv(RULES_CSV, dtype={'antecedent': str, 'consequent': str})
print(f'Tổng số luật: {len(rules):,}')

# Nếu có file tên mặt hàng (ví dụ bộ Instacart trong SPMF), điền vào đây: {mã: 'tên'}
ITEM_NAMES = {}
def name(i): return ITEM_NAMES.get(int(i), f'#{i}')
def side(s): return '{' + ', '.join(name(i) for i in s.split()) + '}'

top = rules.head(10).copy()
top['luật'] = top['antecedent'].map(side) + ' ⇒ ' + top['consequent'].map(side)
top.index = range(1, len(top) + 1)
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
top[['luật', 'count', 'support', 'confidence', 'lift']].round(4)

In [ ]:
if len(top):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(top['luật'][::-1], top['lift'][::-1])
    ax.axvline(1.0, linestyle='--', linewidth=1)
    ax.set_xlabel('lift (đường nét đứt = 1: độc lập)')
    ax.set_title(f'Top-10 luật theo lift (minsup={MINSUP}, minconf={MINCONF})')
    plt.tight_layout(); plt.show()

## 7. Diễn giải ý nghĩa kinh doanh

Ô dưới tự viết một câu diễn giải cho từng luật trong top-10.

In [ ]:
for i, r in top.iterrows():
    print(f"{i:>2}. Khách mua {side(r['antecedent'])} thì {r['confidence']:.0%} cũng mua {side(r['consequent'])}"
          f" (gấp {r['lift']:.2f} lần so với mua ngẫu nhiên; {r['support']:.2%} số giỏ chứa cả hai).")

**Cách đọc các chỉ số**

- **support**: luật áp dụng cho bao nhiêu phần trăm giỏ hàng. Quá thấp thì ít giá trị thực tế.
- **confidence**: nếu giỏ có X thì xác suất có Y. Chỉ số này dễ cao giả khi Y vốn là mặt hàng rất phổ biến.
- **lift**: so với mua ngẫu nhiên. `lift > 1` nghĩa là X và Y đi cùng nhau nhiều hơn kỳ vọng, `= 1` là độc lập, `< 1` là loại trừ nhau.

**Ứng dụng:** gợi ý bán kèm (cross-sell), đặt các mặt hàng liên quan gần nhau, tạo combo khuyến mãi, phân tích tồn kho theo cụm sản phẩm.

**Lưu ý khi viết báo cáo:**

- Retail ẩn danh mã mặt hàng, nên chỉ nói được cấu trúc của luật. Muốn diễn giải bằng tên sản phẩm cần dataset có tên (ví dụ bộ Instacart trong SPMF kèm file tên mặt hàng), rồi điền vào `ITEM_NAMES`.
- Luật chỉ thể hiện **tương quan trong dữ liệu**, không chứng minh nhân quả.
- Kết quả phụ thuộc vào `MINSUP` và `MINCONF`; hãy ghi rõ giá trị đã dùng.

## 8. Kiểm tra tái lập và đối chiếu hai bộ đếm

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()

# (a) Chạy lại bộ sinh luật: file CSV phải giống hệt từng byte
!julia --project=. bin/rules.jl --itemsets {ITEMSETS} --data {DATA} --minconf {MINCONF} --top 0 --out output/retail-rules-rerun.csv > /dev/null
print('rules.csv chạy lại giống hệt:', sha(RULES_CSV) == sha('output/retail-rules-rerun.csv'))

# (b) Bộ đếm scan phải cho đúng cùng tập itemset với hash-tree
!julia --project=. bin/apriori.jl --input {DATA} --output output/retail-scan.txt --minsup {MINSUP} --counter scan --quiet
a = set(open(ITEMSETS).read().splitlines()) - {''}
b = set(open('output/retail-scan.txt').read().splitlines()) - {''}
print('scan và hash-tree cùng kết quả:', a == b, f'({len(a):,} itemsets)')

## 9. Tải kết quả về máy (tùy chọn)

In [ ]:
try:
    from google.colab import files
    files.download(RULES_CSV)
except ImportError:
    print('Không chạy trong Colab, file nằm ở', RULES_CSV)